# LeetCode 3454 — Separate Squares II bằng Python

- Link: https://leetcode.com/problems/separate-squares-ii/
- Độ khó: **Hard**
- Chủ đề: **Quét đường thẳng (sweep line)** theo trục `y`, **cây phân đoạn** tính **độ dài phủ** trên trục `x` (diện tích hợp), nội suy tuyến tính trong dải

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def separateSquares(self, squares: List[List[int]]) -> float:
```

## 1. Mục tiêu học

1. Hiểu "diện tích hợp" (phần chồng chỉ tính một lần) và vì sao hàm "diện tích phía dưới đường `y`" là **liên tục, không giảm, tuyến tính từng khúc**.
2. Quét theo `y`: giữa hai sự kiện liên tiếp, **độ rộng phủ** `W` trên trục `x` không đổi ⇒ dải diện tích `W · Δy`.
3. Cài **cây phân đoạn đếm phủ** (không cần lazy propagation): `cnt[node]` và `cov[node]`.
4. Tìm `y` nhỏ nhất mà diện tích phía dưới đạt **một nửa** tổng: nội suy trong dải, xử lý đúng các dải rỗng (`W = 0`).
5. Dùng số nguyên cho phần tích lũy để tránh sai số.

## 2. Đề bài & ví dụ

`squares[i] = [x, y, l]`: hình vuông góc dưới trái `(x, y)`, cạnh `l`, song song trục. Tìm **`y` nhỏ nhất** của một đường ngang sao cho **diện tích phủ** phía trên bằng phía dưới. Vùng chồng nhau chỉ tính **một lần**. Sai số `10^{-5}` được chấp nhận.

**Ví dụ 1:** `[[0,0,1],[2,2,1]]` → `1.0` (mọi đường trong `[1, 2]` đều chia đôi; nhỏ nhất là 1).

**Ví dụ 2:** `[[0,0,2],[1,1,1]]` → `1.0` (hình nhỏ nằm trong hình lớn).

**Ràng buộc:** `n ≤ 5·10^4`, tọa độ ≤ `10^9`, tổng diện tích ≤ `10^{15}`.

In [1]:
from typing import List

examples = [([[0, 0, 1], [2, 2, 1]], 1.0), ([[0, 0, 2], [1, 1, 1]], 1.0)]
for sq, exp in examples:
    print(sq, '-> kỳ vọng', exp)

[[0, 0, 1], [2, 2, 1]] -> kỳ vọng 1.0
[[0, 0, 2], [1, 1, 1]] -> kỳ vọng 1.0


## 3. Kiến thức nền

### 3.1. Hợp các đoạn trên trục `x`

Độ dài hợp của các đoạn `[a, b)`: sắp theo `a`, gộp các đoạn chồng/chạm nhau.

In [2]:
def union_length(intervals):
    total, cur_s, cur_e = 0, None, None
    for a, b in sorted(intervals):
        if cur_e is None or a > cur_e:
            if cur_e is not None:
                total += cur_e - cur_s
            cur_s, cur_e = a, b
        else:
            cur_e = max(cur_e, b)
    if cur_e is not None:
        total += cur_e - cur_s
    return total

print(union_length([(0, 2), (1, 3), (5, 6)]), union_length([(0, 2), (1, 2)]), union_length([]))

4 2 0


### 3.2. Diện tích phía dưới `y` là hàm tuyến tính từng khúc

Các mốc `y` sự kiện là `y_i` và `y_i + l_i`. Giữa hai mốc liên tiếp `y0 < y1`, tập hình vuông "đang cắt" không đổi ⇒ độ rộng phủ `W` không đổi ⇒ diện tích dưới `y ∈ [y0, y1]` là `A(y0) + W · (y − y0)`.

### 3.3. Cây phân đoạn đếm phủ

Nén tọa độ `x` thành các ô cơ bản `[xs[i], xs[i+1])`. Mỗi nút giữ `cnt` (số lần cả đoạn của nút được phủ **trọn** bởi các cập nhật) và `cov` (độ dài phủ trong nút):

- `cnt > 0` ⇒ `cov` = toàn bộ độ dài nút;
- ngược lại `cov` = tổng `cov` hai con (lá: 0).

Vì mỗi lệnh "+1" luôn có lệnh "−1" tương ứng trên **đúng cùng** các nút, không cần đẩy lazy xuống.

In [3]:
class CoverTree:
    def __init__(self, xs):
        self.xs = xs
        self.m = len(xs) - 1
        self.cnt = [0] * (4 * max(1, self.m))
        self.cov = [0] * (4 * max(1, self.m))

    def update(self, l, r, v, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.m
        if r <= lo or hi <= l:
            return
        if l <= lo and hi <= r:
            self.cnt[node] += v
        else:
            mid = (lo + hi) // 2
            self.update(l, r, v, 2 * node, lo, mid)
            self.update(l, r, v, 2 * node + 1, mid, hi)
        if self.cnt[node]:
            self.cov[node] = self.xs[hi] - self.xs[lo]
        elif hi - lo == 1:
            self.cov[node] = 0
        else:
            self.cov[node] = self.cov[2 * node] + self.cov[2 * node + 1]

    def covered(self):
        return self.cov[1]

t = CoverTree([0, 1, 2, 3, 5, 6])
t.update(0, 2, 1); t.update(1, 3, 1); t.update(4, 5, 1)
print('phủ [0,2)+[1,3)+[5,6):', t.covered())
t.update(1, 3, -1)
print('bỏ [1,3):', t.covered())

phủ [0,2)+[1,3)+[5,6): 4
bỏ [1,3): 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: các dải theo `y`:

| dải | hình đang cắt | W | diện tích dải | tích lũy |
|---|---|---|---|---|
| `[0, 1]` | hình 1 | 1 | 1 | 1 |
| `[1, 2]` | — | 0 | 0 | 1 |
| `[2, 3]` | hình 2 | 1 | 1 | 2 |

Tổng 2, một nửa là 1 ⇒ đạt **đúng** ở cuối dải đầu ⇒ `y = 1` (nhỏ nhất; mọi `y ∈ [1, 2]` cũng được).

Ví dụ 2: dải `[0,1]` W = 2 (diện tích 2), dải `[1,2]` W = 2 (hình nhỏ nằm trong hình lớn), tổng 4, nửa là 2 ⇒ `y = 1`.

Ví dụ thêm `[[0,0,2]]`: một dải `[0,2]` W = 2, tổng 4, nửa 2 ⇒ nội suy `y = 0 + 2/2 = 1`.

## 5. Cách 1 — Brute Force: đếm ô vuông đơn vị (tọa độ nhỏ)

Khi tọa độ nhỏ, đánh dấu mọi ô đơn vị bị phủ; mỗi hàng ô `[r, r+1)` có độ rộng phủ = số ô của hàng. Dùng `Fraction` để kết quả chính xác.

In [4]:
from fractions import Fraction

def separate_brute(squares):
    cells = set()
    for x, y, l in squares:
        for i in range(x, x + l):
            for j in range(y, y + l):
                cells.add((i, j))
    rows = {}
    for i, j in cells:
        rows[j] = rows.get(j, 0) + 1
    T, cum = len(cells), 0
    for r in sorted(rows):
        w = rows[r]
        if 2 * (cum + w) >= T:
            return r + Fraction(T - 2 * cum, 2 * w)
        cum += w

for sq, exp in examples:
    print(separate_brute(sq), '| kỳ vọng', exp)
print('[[0,0,2]] ->', separate_brute([[0, 0, 2]]))

1 | kỳ vọng 1.0
1 | kỳ vọng 1.0
[[0,0,2]] -> 1


**Độ phức tạp:** `O(Σ l²)` — chỉ dùng cho tọa độ rất nhỏ (oracle chính xác tuyệt đối).

## 6. Ý tưởng thuật toán / bước đột phá

1. Sự kiện: `(y, +1, x, x + l)` và `(y + l, −1, x, x + l)`; sắp theo `y`.
2. Quét: trước khi xử lý các sự kiện tại `y` mới, ghi dải `(y_prev, y, W)` với `W = cov[1]` hiện tại.
3. Tổng `T = Σ W · Δy` (số nguyên). Duyệt dải với `W > 0`: dải đầu tiên mà `2 · (cum + W · Δy) ≥ T` chứa đáp án ⇒ `y = y0 + (T − 2 · cum) / (2W)`.

Bỏ qua dải `W = 0` khi tìm đáp án: nếu tích lũy đạt đúng nửa tại cuối một dải có `W > 0` thì công thức trả về `y1` của dải đó — **trước** mọi dải rỗng phía sau ⇒ là `y` nhỏ nhất.

In [5]:
def answer_from_strips(strips):
    T = sum((y1 - y0) * W for y0, y1, W in strips)
    cum = 0
    for y0, y1, W in strips:
        a = (y1 - y0) * W
        if W and 2 * (cum + a) >= T:
            return y0 + (T - 2 * cum) / (2 * W)
        cum += a
    return float(strips[-1][1]) if strips else 0.0

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quét, mỗi dải tính lại độ rộng hợp bằng sắp xếp

In [6]:
def strips_naive(squares):
    ev = sorted({y for x, y, l in squares} | {y + l for x, y, l in squares})
    res = []
    for y0, y1 in zip(ev, ev[1:]):
        W = union_length([(x, x + l) for x, y, l in squares if y <= y0 and y + l >= y1])
        res.append((y0, y1, W))
    return res

def separate_naive(squares):
    return answer_from_strips(strips_naive(squares))

for sq, exp in examples:
    print(separate_naive(sq), '| kỳ vọng', exp)

1.0 | kỳ vọng 1.0
1.0 | kỳ vọng 1.0


`O(n)` dải × `O(n log n)` mỗi dải ⇒ `O(n² log n)` — dùng cho `n` vài trăm.

### 7.2. Cách 3 — Quét + cây phân đoạn đếm phủ (lời giải chính)

In [7]:
def strips_segtree(squares):
    xs = sorted({x for x, y, l in squares} | {x + l for x, y, l in squares})
    idx = {v: i for i, v in enumerate(xs)}
    tree = CoverTree(xs)
    events = []
    for x, y, l in squares:
        events.append((y, 1, idx[x], idx[x + l]))
        events.append((y + l, -1, idx[x], idx[x + l]))
    events.sort()
    strips, prev = [], events[0][0]
    for y, v, l, r in events:
        if y > prev:
            strips.append((prev, y, tree.covered()))
            prev = y
        tree.update(l, r, v)
    return strips

def separate_segtree(squares):
    return answer_from_strips(strips_segtree(squares))

for sq, exp in examples:
    print(separate_segtree(sq), '| kỳ vọng', exp)

1.0 | kỳ vọng 1.0
1.0 | kỳ vọng 1.0


### 7.3. Cách 4 — Tìm kiếm nhị phân trên `y` với hàm "diện tích phía dưới"

Từ danh sách dải, `A(y)` tính bằng `bisect` + tổng tiền tố. `A` không giảm ⇒ nhị phân số thực tìm `y` nhỏ nhất với `2A(y) ≥ T` (100 vòng lặp). Cách này minh họa **tính đơn điệu**; cách 3 nội suy trực tiếp chính xác hơn.

In [8]:
import bisect

def separate_bsearch(squares, iters=100):
    strips = strips_segtree(squares)
    starts = [s[0] for s in strips]
    pref = [0]
    for y0, y1, W in strips:
        pref.append(pref[-1] + (y1 - y0) * W)
    T = pref[-1]

    def area_below(y):
        k = bisect.bisect_right(starts, y) - 1
        if k < 0:
            return 0
        y0, y1, W = strips[k]
        return pref[k] + W * (min(y, y1) - y0)

    lo, hi = float(strips[0][0]), float(strips[-1][1])
    for _ in range(iters):
        mid = (lo + hi) / 2
        if 2 * area_below(mid) >= T:
            hi = mid
        else:
            lo = mid
    return hi

for sq, exp in examples:
    print(round(separate_bsearch(sq), 9), '| kỳ vọng', exp)

1.0 | kỳ vọng 1.0
1.0 | kỳ vọng 1.0


## 8. TRACE từng bước

Trace cách 3 trên `[[0,0,2],[1,1,2],[5,0,1]]`: in sự kiện, độ rộng phủ và dải.

In [9]:
def trace(squares):
    xs = sorted({x for x, y, l in squares} | {x + l for x, y, l in squares})
    idx = {v: i for i, v in enumerate(xs)}
    tree = CoverTree(xs)
    events = sorted([(y, 1, x, x + l) for x, y, l in squares] + [(y + l, -1, x, x + l) for x, y, l in squares])
    print('tọa độ x nén:', xs)
    strips, prev = [], events[0][0]
    for y, v, a, b in events:
        if y > prev:
            strips.append((prev, y, tree.covered()))
            print(f"  dải y ∈ [{prev}, {y}]: W = {tree.covered()}, diện tích {tree.covered() * (y - prev)}")
            prev = y
        tree.update(idx[a], idx[b], v)
        print(f"sự kiện y={y}: {'thêm' if v == 1 else 'bỏ'} [{a}, {b}) -> phủ {tree.covered()}")
    ans = answer_from_strips(strips)
    print('tổng =', sum((y1 - y0) * W for y0, y1, W in strips), '| đáp án y =', ans, '| vét cạn:', separate_brute(squares))

trace([[0, 0, 2], [1, 1, 2], [5, 0, 1]])

tọa độ x nén: [0, 1, 2, 3, 5, 6]
sự kiện y=0: thêm [0, 2) -> phủ 2
sự kiện y=0: thêm [5, 6) -> phủ 3
  dải y ∈ [0, 1]: W = 3, diện tích 3
sự kiện y=1: bỏ [5, 6) -> phủ 2
sự kiện y=1: thêm [1, 3) -> phủ 3
  dải y ∈ [1, 2]: W = 3, diện tích 3
sự kiện y=2: bỏ [0, 2) -> phủ 2
  dải y ∈ [2, 3]: W = 2, diện tích 2
sự kiện y=3: bỏ [1, 3) -> phủ 0
tổng = 8 | đáp án y = 1.3333333333333333 | vét cạn: 4/3


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def separateSquares(self, squares: List[List[int]]) -> float:
        xs = sorted({x for x, _, l in squares} | {x + l for x, _, l in squares})
        idx = {v: i for i, v in enumerate(xs)}
        m = len(xs) - 1
        cnt = [0] * (4 * m)
        cov = [0] * (4 * m)

        def update(node, lo, hi, l, r, v):
            if r <= lo or hi <= l:
                return
            if l <= lo and hi <= r:
                cnt[node] += v
            else:
                mid = (lo + hi) // 2
                update(2 * node, lo, mid, l, r, v)
                update(2 * node + 1, mid, hi, l, r, v)
            if cnt[node]:
                cov[node] = xs[hi] - xs[lo]
            elif hi - lo == 1:
                cov[node] = 0
            else:
                cov[node] = cov[2 * node] + cov[2 * node + 1]

        events = []
        for x, y, l in squares:
            events.append((y, 1, idx[x], idx[x + l]))
            events.append((y + l, -1, idx[x], idx[x + l]))
        events.sort()
        strips = []
        prev = events[0][0]
        for y, v, l, r in events:
            if y > prev:
                strips.append((prev, y, cov[1]))
                prev = y
            update(1, 0, m, l, r, v)
        total = sum((y1 - y0) * w for y0, y1, w in strips)
        cum = 0
        for y0, y1, w in strips:
            a = (y1 - y0) * w
            if w and 2 * (cum + a) >= total:
                return y0 + (total - 2 * cum) / (2 * w)
            cum += a
        return float(strips[-1][1])

for sq, exp in examples:
    got = Solution().separateSquares(sq)
    print(got, 'OK' if abs(got - exp) < 1e-5 else 'SAI')

1.0 OK
1.0 OK


## 10. Vì sao thuật toán đúng

1. **Dải đúng:** giữa hai mốc sự kiện liên tiếp, tập hình vuông cắt đường ngang không đổi, nên mặt cắt (hợp các đoạn `x`) không đổi; diện tích dải = `W · Δy`.
2. **Cây phân đoạn đúng:** bất biến `cov[node]` = độ dài phần của nút được phủ bởi các cập nhật đang hoạt động **nằm trong cây con của nút**. Mỗi cập nhật `±1` dừng ở cùng tập nút (phân rã chuẩn của đoạn), nên `cnt` không bao giờ âm và bất biến được giữ khi tính lại từ dưới lên.
3. **Đáp án:** `A(y)` liên tục, không giảm, tăng thực sự trong các dải `W > 0`. `y` nhỏ nhất với `2A(y) = T` nằm trong dải `W > 0` đầu tiên có `2A(y1) ≥ T`; trong dải đó `A` tuyến tính ⇒ nghiệm duy nhất `y0 + (T − 2cum)/(2W)`.
4. **Chính xác số học:** `T`, `cum` là số nguyên (≤ `10^{15}`), chỉ phép chia cuối là số thực ⇒ sai số rất nhỏ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Ô đơn vị | `O(Σ l²)` | `O(Σ l²)` |
| 2. Quét + hợp đoạn mỗi dải | `O(n² log n)` | `O(n)` |
| 3. Quét + cây phân đoạn | `O(n log n)` | `O(n)` |
| 4. Nhị phân trên `y` | `O(n log n + iters · log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'naive': separate_naive, 'segtree': separate_segtree, 'bsearch': separate_bsearch,
         'submit': lambda sq: Solution().separateSquares(sq)}
edge = [
    ([[0, 0, 2]], 1.0),
    ([[0, 0, 1], [0, 0, 1]], 0.5),                 # trùng hoàn toàn
    ([[0, 0, 4], [1, 1, 2]], 2.0),                 # lồng nhau
    ([[0, 0, 1], [5, 10, 1]], 1.0),                # hai hình xa nhau: y nhỏ nhất là 1
    ([[0, 0, 2], [1, 1, 2], [5, 0, 1]], None),
]
for sq, exp in examples + edge:
    b = float(separate_brute(sq))
    if exp is not None:
        assert abs(b - exp) < 1e-12, (sq, b)
    for name, f in impls.items():
        assert abs(f(sq) - b) < 1e-6, (name, sq, f(sq), b)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn ô đơn vị

In [12]:
import random
rng = random.Random(3454)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 6)
    sq = [[rng.randint(0, 8), rng.randint(0, 8), rng.randint(1, 5)] for _ in range(n)]
    b = float(separate_brute(sq))
    for name, f in impls.items():
        assert abs(f(sq) - b) < 1e-6, (name, sq)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

# cỡ vừa: cách 2 vs cách 3 với tọa độ lớn
for _ in range(100):
    n = rng.randint(1, 60)
    sq = [[rng.randint(0, 10 ** 6), rng.randint(0, 10 ** 6), rng.randint(1, 10 ** 5)] for _ in range(n)]
    assert abs(separate_naive(sq) - Solution().separateSquares(sq)) < 1e-6
print('Cách 2 và cách 3 khớp trên 100 bộ tọa độ lớn.')

Khớp vét cạn trên 1500 bộ ngẫu nhiên.


Cách 2 và cách 3 khớp trên 100 bộ tọa độ lớn.


### 12.3. Ca lớn `n = 5·10^4`

In [13]:
import time
n = 50000
cases = {
    'ngẫu nhiên, cạnh nhỏ': [[rng.randint(0, 10 ** 9), rng.randint(0, 10 ** 9), rng.randint(1, 10 ** 5)] for _ in range(n)],
    'chồng chéo dày': [[rng.randint(0, 10 ** 5), rng.randint(0, 10 ** 5), rng.randint(1, 10 ** 5)] for _ in range(n)],
    'lồng nhau': [[i, i, 10 ** 9 - 2 * i] for i in range(n)],
}
for name, sq in cases.items():
    t = time.time(); a = Solution().separateSquares(sq); t1 = time.time() - t
    print(f"{name:22s}: y = {a:.6f} | {t1:.2f}s")

ngẫu nhiên, cạnh nhỏ  : y = 500747306.448776 | 0.70s


chồng chéo dày        : y = 99446.107525 | 1.24s


lồng nhau             : y = 500000000.000000 | 0.90s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng diện tích từng hình (đếm phần chồng nhiều lần) — lời giải của bản I

In [14]:
def wrong_sum_areas(squares):
    T = sum(l * l for _, _, l in squares)
    lo, hi = min(y for _, y, _ in squares), max(y + l for _, y, l in squares)
    def below(h):
        return sum(l * min(max(h - y, 0), l) for _, y, l in squares)
    for _ in range(100):
        mid = (lo + hi) / 2
        if 2 * below(mid) >= T:
            hi = mid
        else:
            lo = mid
    return hi

sq = examples[1][0]                                  # ví dụ 2: hình nhỏ nằm trong hình lớn
print('sai :', round(wrong_sum_areas(sq), 6), '| đúng:', Solution().separateSquares(sq), '| vét cạn:', float(separate_brute(sq)))

sai : 1.166667 | đúng: 1.0 | vét cạn: 1.0


**Vì sao sai:** bản II yêu cầu vùng chồng chỉ tính **một lần** (diện tích hợp). Cộng riêng từng hình làm phần chồng bị tính gấp đôi.

### Lỗi 2 — Trả về mốc sự kiện thay vì nội suy trong dải

In [15]:
def wrong_no_interp(squares):
    strips = strips_segtree(squares)
    T = sum((y1 - y0) * W for y0, y1, W in strips)
    cum = 0
    for y0, y1, W in strips:
        cum += (y1 - y0) * W
        if 2 * cum >= T:
            return float(y1)                       # SAI: không nội suy

print('sai :', wrong_no_interp([[0, 0, 2]]), '| đúng:', Solution().separateSquares([[0, 0, 2]]))

sai : 2.0 | đúng: 1.0


**Vì sao sai:** đường chia đôi thường nằm **giữa** một dải, không trùng mốc sự kiện.

### Lỗi 3 — So sánh nghiêm ngặt `>` nên bỏ qua trường hợp đạt đúng một nửa ở cuối dải

In [16]:
def wrong_strict(squares):
    strips = strips_segtree(squares)
    T = sum((y1 - y0) * W for y0, y1, W in strips)
    cum = 0
    for y0, y1, W in strips:
        a = (y1 - y0) * W
        if W and 2 * (cum + a) > T:                # SAI: phải là >=
            return y0 + (T - 2 * cum) / (2 * W)
        cum += a

sq = examples[0][0]
print('sai :', wrong_strict(sq), '| đúng:', Solution().separateSquares(sq))

sai : 2.0 | đúng: 1.0


**Vì sao sai:** ở ví dụ 1, tích lũy đạt **đúng** một nửa tại `y = 1` (cuối dải đầu). Dùng `>` nhảy qua dải rỗng `[1, 2]` và trả về `2` — vẫn chia đôi nhưng **không nhỏ nhất**.

### Lỗi 4 — Cây phân đoạn: khi `cnt` về 0 quên tính lại từ hai con

In [17]:
def make_wrong_tree_solver():
    def solve(squares):
        xs = sorted({x for x, _, l in squares} | {x + l for x, _, l in squares})
        idx = {v: i for i, v in enumerate(xs)}
        m = len(xs) - 1
        cnt = [0] * (4 * m); cov = [0] * (4 * m)
        def update(node, lo, hi, l, r, v):
            if r <= lo or hi <= l:
                return
            if l <= lo and hi <= r:
                cnt[node] += v
                cov[node] = xs[hi] - xs[lo] if cnt[node] else 0      # SAI: phải lấy tổng hai con
                return
            mid = (lo + hi) // 2
            update(2 * node, lo, mid, l, r, v); update(2 * node + 1, mid, hi, l, r, v)
            cov[node] = xs[hi] - xs[lo] if cnt[node] else cov[2 * node] + cov[2 * node + 1]
        events = sorted([(y, 1, idx[x], idx[x + l]) for x, y, l in squares] +
                        [(y + l, -1, idx[x], idx[x + l]) for x, y, l in squares])
        strips, prev = [], events[0][0]
        for y, v, l, r in events:
            if y > prev:
                strips.append((prev, y, cov[1])); prev = y
            update(1, 0, m, l, r, v)
        return answer_from_strips(strips)
    return solve

wrong_tree = make_wrong_tree_solver()
rng = random.Random(4)
ce = None
for _ in range(20000):
    sq = [[rng.randint(0, 5), rng.randint(0, 5), rng.randint(1, 4)] for _ in range(rng.randint(2, 4))]
    if abs(wrong_tree(sq) - float(separate_brute(sq))) > 1e-9:
        ce = sq; break
print('phản ví dụ:', ce)
print('sai :', wrong_tree(ce), '| đúng:', float(separate_brute(ce)))

phản ví dụ: [[2, 5, 3], [0, 4, 3], [5, 3, 2]]
sai : 5.5 | đúng: 5.6


**Vì sao sai:** khi một nút hết bị phủ trọn (`cnt = 0`), vẫn có thể còn các hình nhỏ hơn phủ **một phần** nút (lưu ở các con). Phải lấy `cov = cov[trái] + cov[phải]` (trừ khi là lá).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Diện tích hợp các hình chữ nhật" ⇒ sweep line + cây phân đoạn đếm phủ (LC 850).
- "Tìm đường chia đôi" ⇒ hàm tích lũy đơn điệu ⇒ nội suy trong khúc tuyến tính (hoặc nhị phân).
- Giữ tích lũy bằng **số nguyên**, chỉ chia ở bước cuối.
- Đề hỏi giá trị **nhỏ nhất** ⇒ cẩn thận với so sánh `≥` và các dải rỗng.

## 15. Biến thể của bài

1. **LC 3453 — Separate Squares I:** phần chồng tính nhiều lần ⇒ không cần cây phân đoạn (hàm `below` cộng từng hình).
2. **LC 850 — Rectangle Area II:** chỉ cần tổng diện tích hợp (mod `10^9 + 7`).
3. Chia theo tỉ lệ `p : q` thay vì `1 : 1` ⇒ thay `2A ≥ T` bằng `(p + q) A ≥ p T`.
4. Đường **dọc** chia đôi ⇒ hoán đổi vai trò `x`, `y`.

In [18]:
# Biến thể 2 (LC 850) và biến thể 3 (chia theo tỉ lệ p : q)
def union_area(rects):
    sq_like = rects
    xs = sorted({x1 for x1, y1, x2, y2 in rects} | {x2 for x1, y1, x2, y2 in rects})
    idx = {v: i for i, v in enumerate(xs)}
    tree = CoverTree(xs)
    events = sorted([(y1, 1, idx[x1], idx[x2]) for x1, y1, x2, y2 in rects] +
                    [(y2, -1, idx[x1], idx[x2]) for x1, y1, x2, y2 in rects])
    area, prev = 0, events[0][0]
    for y, v, l, r in events:
        area += tree.covered() * (y - prev)
        prev = y
        tree.update(l, r, v)
    return area

assert union_area([[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]]) == 6              # LC 850 ví dụ 1
assert union_area([[0, 0, 10 ** 9, 10 ** 9]]) == 10 ** 18

def split_ratio(squares, p, q):
    strips = strips_segtree(squares)
    T = sum((y1 - y0) * W for y0, y1, W in strips)
    cum = 0
    for y0, y1, W in strips:
        a = (y1 - y0) * W
        if W and (p + q) * (cum + a) >= p * T:
            return y0 + (p * T - (p + q) * cum) / ((p + q) * W)
        cum += a

assert abs(split_ratio([[0, 0, 4]], 1, 3) - 1.0) < 1e-12
assert all(abs(split_ratio(sq, 1, 1) - Solution().separateSquares(sq)) < 1e-9 for sq, _ in examples)
print('LC 850 ví dụ: 6; chia 1:3 hình [[0,0,4]] tại y =', split_ratio([[0, 0, 4]], 1, 3))

LC 850 ví dụ: 6; chia 1:3 hình [[0,0,4]] tại y = 1.0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Ô đơn vị | `O(Σ l²)` | Oracle chính xác |
| 2 | Quét + hợp đoạn mỗi dải | `O(n² log n)` | Đơn giản |
| 3 | Quét + cây phân đoạn | `O(n log n)` | **Nộp bài** |
| 4 | Nhị phân trên `y` | `O(n log n)` | Minh họa tính đơn điệu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[0, 0, 3], [0, 0, 1]]`.

**Bài 2 (tính tay):** `[[0, 0, 2], [2, 0, 2]]` (hai hình kề nhau theo `x`).

**Bài 3 (điền TODO):** viết `separate_squares_I(squares)` — bản I (chồng tính nhiều lần), không cần cây phân đoạn.

**Bài 4 (tự viết lại):** đường **dọc** chia đôi diện tích hợp.

**Bài 5 (mini challenge):** cài cây phân đoạn **không đệ quy** (mảng, duyệt từ dưới lên) và so tốc độ.

In [19]:
def separate_squares_I_todo(squares):
    # TODO
    raise NotImplementedError

def vertical_split_todo(squares):
    # TODO
    raise NotImplementedError

def iterative_cover_tree_todo(xs):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hình nhỏ nằm trong hình lớn. (2) Chỉ còn hình `3 × 3`. (3) `y = 1.5`.

**Bài 2.** (1) Một dải `[0, 2]` với `W = 4`. (2) Tổng 8. (3) `y = 1`.

**Bài 3.** (1) `below(h) = Σ l · clamp(h − y, 0, l)`. (2) Hàm tuyến tính từng khúc với mốc `y`, `y + l`. (3) Quét mốc, nội suy như bản II.

**Bài 4.** (1) Đổi `(x, y)` thành `(y, x)`. (2) Gọi lại lời giải. (3) Kiểm tra bằng vét cạn ô đơn vị theo cột.

**Bài 5.** (1) Mảng kích thước `2·size`. (2) Cập nhật các nút phủ trọn bằng hai con trỏ `l, r` đi lên. (3) Sau đó tính lại `cov` dọc đường từ hai lá biên lên gốc.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
a1 = Solution().separateSquares([[0, 0, 3], [0, 0, 1]])
a2 = Solution().separateSquares([[0, 0, 2], [2, 0, 2]])
assert abs(a1 - 1.5) < 1e-12 and abs(a2 - 1.0) < 1e-12
print('Bài 1:', a1, '| Bài 2:', a2)

# Bài 3 — bản I bằng quét mốc
def separate_squares_I(squares):
    T = sum(l * l for _, _, l in squares)
    ev = {}
    for _, y, l in squares:                     # độ rộng tổng (đếm lặp) thay đổi tại y (+l) và y+l (-l)
        ev[y] = ev.get(y, 0) + l
        ev[y + l] = ev.get(y + l, 0) - l
    ys = sorted(ev)
    W, cum = 0, 0
    for y0, y1 in zip(ys, ys[1:]):
        W += ev[y0]
        a = W * (y1 - y0)
        if W and 2 * (cum + a) >= T:
            return y0 + (T - 2 * cum) / (2 * W)
        cum += a

for sq in ([[0, 0, 1], [2, 2, 1]], [[0, 0, 2], [1, 1, 1]]):
    print('bản I:', sq, '->', separate_squares_I(sq))
assert abs(separate_squares_I([[0, 0, 2], [1, 1, 1]]) - 7 / 6) < 1e-12     # ví dụ 2 của LC 3453

Bài 1: 1.5 | Bài 2: 1.0
bản I: [[0, 0, 1], [2, 2, 1]] -> 1.0
bản I: [[0, 0, 2], [1, 1, 1]] -> 1.1666666666666667


In [21]:
# Bài 4 — đường dọc
def vertical_split(squares):
    return Solution().separateSquares([[y, x, l] for x, y, l in squares])

def vertical_brute(squares):
    return separate_brute([[y, x, l] for x, y, l in squares])

rng = random.Random(19)
for _ in range(300):
    sq = [[rng.randint(0, 8), rng.randint(0, 8), rng.randint(1, 4)] for _ in range(rng.randint(1, 5))]
    assert abs(vertical_split(sq) - float(vertical_brute(sq))) < 1e-9
print('Bài 4 khớp vét cạn; ví dụ 1 theo chiều dọc:', vertical_split(examples[0][0]))

Bài 4 khớp vét cạn; ví dụ 1 theo chiều dọc: 1.0


In [22]:
# Bài 5 — cây phân đoạn không đệ quy (đếm phủ), dùng mảng dựa trên lá
class IterCoverTree:
    def __init__(self, xs):
        self.xs = xs
        self.m = len(xs) - 1
        size = 1
        while size < self.m:
            size *= 2
        self.size = size
        self.cnt = [0] * (2 * size)
        self.cov = [0] * (2 * size)
        self.length = [0] * (2 * size)
        for i in range(self.m):
            self.length[size + i] = xs[i + 1] - xs[i]
        for i in range(size - 1, 0, -1):
            self.length[i] = self.length[2 * i] + self.length[2 * i + 1]

    def _pull(self, i):
        if self.cnt[i]:
            self.cov[i] = self.length[i]
        elif i >= self.size:
            self.cov[i] = 0
        else:
            self.cov[i] = self.cov[2 * i] + self.cov[2 * i + 1]

    def update(self, l, r, v):
        l0, r0 = l + self.size, r - 1 + self.size
        lo, hi = l0, r + self.size
        while lo < hi:
            if lo & 1:
                self.cnt[lo] += v; self._pull(lo); lo += 1
            if hi & 1:
                hi -= 1; self.cnt[hi] += v; self._pull(hi)
            lo //= 2; hi //= 2
        for leaf in (l0, r0):
            i = leaf // 2
            while i:
                self._pull(i); i //= 2

    def covered(self):
        return self.cov[1]

def separate_iter(squares):
    xs = sorted({x for x, _, l in squares} | {x + l for x, _, l in squares})
    idx = {v: i for i, v in enumerate(xs)}
    tree = IterCoverTree(xs)
    events = sorted([(y, 1, idx[x], idx[x + l]) for x, y, l in squares] +
                    [(y + l, -1, idx[x], idx[x + l]) for x, y, l in squares])
    strips, prev = [], events[0][0]
    for y, v, l, r in events:
        if y > prev:
            strips.append((prev, y, tree.covered())); prev = y
        tree.update(l, r, v)
    return answer_from_strips(strips)

rng = random.Random(5)
for _ in range(500):
    sq = [[rng.randint(0, 8), rng.randint(0, 8), rng.randint(1, 5)] for _ in range(rng.randint(1, 6))]
    assert abs(separate_iter(sq) - float(separate_brute(sq))) < 1e-9
sq = cases['chồng chéo dày']
t = time.time(); a = separate_iter(sq); t1 = time.time() - t
t = time.time(); b = Solution().separateSquares(sq); t2 = time.time() - t
assert abs(a - b) < 1e-6
print(f'Bài 5 khớp vét cạn; n = 5·10^4: không đệ quy {t1:.2f}s, đệ quy {t2:.2f}s')

Bài 5 khớp vét cạn; n = 5·10^4: không đệ quy 0.88s, đệ quy 1.18s


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
xs = sorted(mọi x và x+l); nén chỉ số
events = (y, +1, l, r), (y+l, -1, l, r); sort
for y, v, l, r in events:
    if y > prev: strips.append((prev, y, cov[1])); prev = y
    update(l, r, v)          # cnt[node] += v; cov = full nếu cnt else tổng con (lá: 0)
T = Σ W·Δy; cum = 0
for (y0, y1, W): if W and 2(cum + W·Δy) >= T: return y0 + (T - 2cum)/(2W)
```

**Checklist:**

- [ ] Diện tích **hợp** (không cộng từng hình)?
- [ ] Nội suy trong dải?
- [ ] So sánh `≥` để lấy `y` nhỏ nhất, bỏ qua dải `W = 0`?
- [ ] Cây phân đoạn: `cnt = 0` thì lấy tổng hai con?
- [ ] Tích lũy bằng số nguyên?